# UU 历史价格与在售数量

入口版本 uu-history-notebook-1.1，交付时未执行。先在自己已登录的 Edge UU 页面运行浏览器代码，下载结果，再使用本 Notebook 纯离线入库。不会启动原连续观察，也不读取登录凭证。

每轮一个商品、价格7/30/90天/半年，数量7/15/30天，共7组；真实稳定性仍待本地试跑验收。

1.1修复：两种指标窗口不同，旧15天价格请求不再发送。已下载的1.0批次仍可导入；浏览器面板可载入旧批次，保留成功数据并迁移至正确7组。先刷新UU页面并替换代码段，再载入批次、正常选择半年价格、人工确认开始。

In [ ]:
from pathlib import Path
import sys, json, subprocess
from datetime import datetime, timezone, timedelta

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/import_uu_chart_batch.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("请从 CS_YouOweMe 项目或它的 Notebook 目录打开。")
print("浏览器采集代码：", ROOT / "scripts/collect_uu_charts.js")
print("完整使用说明：", ROOT / "docs/usage/UU_HISTORY_COLLECT.md")


## 先完成浏览器一轮

1. Edge正常登录UU，打开精确商品磨损页。
2. F12 → Sources → Snippets，新建代码段，把上面 collect_uu_charts.js 文件全文复制进去并运行。
3. 在原图表选择成交价格的半年，等待正常返回；在右上角面板确认模板，填写商品规格，点击开始。
4. 完成或停止后保存下载的 uu_charts_*.json。限流或登录失效会停止，不反复重试。

下面只选择下载文件并在本地入库。

In [ ]:
from tkinter import Tk, filedialog

chooser = Tk()
chooser.withdraw()
try:
    chosen = filedialog.askopenfilename(title="选择 UU 浏览器采集的批次 JSON", filetypes=[("JSON", "*.json")])
finally:
    chooser.destroy()
BATCH_FILE = Path(chosen) if chosen else None
print("已选择批次文件。" if BATCH_FILE else "已取消；尚未导入。")


In [ ]:
if BATCH_FILE is None or not BATCH_FILE.is_file():
    raise RuntimeError("请先选择已下载的批次JSON。")
run_stamp = datetime.now(timezone(timedelta(hours=8))).strftime("%Y%m%dT%H%M%S%f+0800")
OUTPUT = ROOT / "data/local/staging/uu_charts" / ("browser_import_" + run_stamp)
result = subprocess.run([sys.executable, "-B", str(ROOT / "scripts/import_uu_chart_batch.py"), str(BATCH_FILE), "--output", str(OUTPUT)], cwd=ROOT, text=True, capture_output=True)
print(result.stdout.strip())
if result.returncode:
    raise RuntimeError("离线入库未通过校验，没有打印输入内容；请核对批次和工具版本。")
print("结果目录：", OUTPUT)


In [ ]:
report = json.loads((OUTPUT / "report.json").read_text(encoding="utf-8"))
print("当前计划全部完成：", report["complete"], "；状态：", report["status_counts"])
for task in report["tasks"]:
    summary = task.get("summary", {})
    print(task["task_id"], task["status"], "点数", summary.get("rows"), "固定间隔毫秒", summary.get("constant_interval_ms"))
print("数据库：", OUTPUT / "chart_batch.sqlite")
print("表格：", OUTPUT / "points.csv")
print("完整报告：", OUTPUT / "report.json")


当前仍为L0来源观察。用户填写的商品描述不等于完成身份核验，浏览器收取时间也不等于历史发布时刻。数据按各窗口独立保存，不用当前盘口回填历史；当前报价既支持决策，也为未来训练积累特征。